# 04 - Global Market Data Ingestion

## Objective
Download global market proxy data and create daily return features to be merged with stock dataset.

## Data Source
Yahoo Finance using yfinance library

## Date Range
2023-01-01 to 2026-01-01

## Tickers
| Ticker | Description |
|--------|-------------|
| ^GSPC  | S&P 500 |
| ^IXIC  | NASDAQ |
| ^DJI   | Dow Jones |
| GC=F   | Gold |
| CL=F   | Crude Oil |
| INR=X  | USD/INR |
| ^VIX   | Volatility Index |
| ^NSEI  | Nifty Index |

## 1. Import Libraries

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')

## 2. Configuration

In [2]:
# Date range as per project requirements
START_DATE = "2023-01-01"
END_DATE = "2026-01-01"

# Tickers to download with their column names
TICKERS = {
    "^GSPC": "SP500",
    "^IXIC": "NASDAQ",
    "^DJI": "DOW",
    "GC=F": "GOLD",
    "CL=F": "OIL",
    "INR=X": "USDINR",
    "^VIX": "VIX",
    "^NSEI": "NIFTY"
}

# Output path
OUTPUT_DIR = "../Market_Data/raw"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "global_market_data.csv")

print(f"Date Range: {START_DATE} to {END_DATE}")
print(f"Tickers to download: {list(TICKERS.keys())}")
print(f"Output file: {OUTPUT_FILE}")

Date Range: 2023-01-01 to 2026-01-01
Tickers to download: ['^GSPC', '^IXIC', '^DJI', 'GC=F', 'CL=F', 'INR=X', '^VIX', '^NSEI']
Output file: ../Market_Data/raw\global_market_data.csv


## 3. Download Close Prices for All Tickers

In [3]:
def download_close_prices(tickers: dict, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Download Close prices for all tickers from Yahoo Finance.
    
    Args:
        tickers: Dictionary mapping ticker symbols to column names
        start_date: Start date in YYYY-MM-DD format
        end_date: End date in YYYY-MM-DD format
    
    Returns:
        DataFrame with Date index and Close prices for each ticker
    """
    close_prices = pd.DataFrame()
    
    for ticker, name in tickers.items():
        print(f"Downloading {ticker} ({name})...")
        try:
            data = yf.download(ticker, start=start_date, end=end_date, progress=False)
            if not data.empty:
                # Handle multi-level columns if present
                if isinstance(data.columns, pd.MultiIndex):
                    close = data['Close'][ticker] if ticker in data['Close'].columns else data['Close'].iloc[:, 0]
                else:
                    close = data['Close']
                close_prices[name] = close
                print(f"  Downloaded {len(close)} records")
            else:
                print(f"  No data available for {ticker}")
        except Exception as e:
            print(f"  Error downloading {ticker}: {e}")
    
    return close_prices

# Download close prices
close_prices_df = download_close_prices(TICKERS, START_DATE, END_DATE)
print(f"\nDownloaded data shape: {close_prices_df.shape}")

  Downloaded 752 records
  Downloaded 752 records
  Downloaded 752 records
  Downloaded 754 records
  Downloaded 754 records
  Downloaded 779 records
  Downloaded 752 records
  Downloaded 740 records

Downloaded data shape: (752, 8)


In [4]:
# Check downloaded data
print("Close Prices Preview:")
print(close_prices_df.head())
print(f"\nDate range: {close_prices_df.index.min()} to {close_prices_df.index.max()}")

Close Prices Preview:
                  SP500        NASDAQ           DOW         GOLD        OIL  \
Date                                                                          
2023-01-03  3824.139893  10386.980469  33136.371094  1839.699951  76.930000   
2023-01-04  3852.969971  10458.759766  33269.769531  1852.800049  72.839996   
2023-01-05  3808.100098  10305.240234  32930.078125  1834.800049  73.669998   
2023-01-06  3895.080078  10569.290039  33630.609375  1864.199951  73.769997   
2023-01-09  3892.090088  10635.650391  33517.648438  1872.699951  74.629997   

               USDINR        VIX         NIFTY  
Date                                            
2023-01-03  82.706001  22.900000  18232.550781  
2023-01-04  82.785698  22.010000  18042.949219  
2023-01-05  82.666496  22.459999  17992.150391  
2023-01-06  82.645203  21.129999  17859.449219  
2023-01-09  82.273399  21.969999  18101.199219  

Date range: 2023-01-03 00:00:00 to 2025-12-31 00:00:00


## 4. Convert Close Prices to Daily Returns

In [5]:
def calculate_daily_returns(close_prices: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate daily returns from close prices using pct_change().
    
    Args:
        close_prices: DataFrame with Close prices
    
    Returns:
        DataFrame with daily returns
    """
    # Calculate percentage change (daily returns)
    returns = close_prices.pct_change()
    
    # Rename columns to include _RET suffix
    returns.columns = [f"{col}_RET" for col in returns.columns]
    
    return returns

# Calculate daily returns
returns_df = calculate_daily_returns(close_prices_df)
print("Daily Returns Preview (with NaN from pct_change):")
print(returns_df.head())

Daily Returns Preview (with NaN from pct_change):
            SP500_RET  NASDAQ_RET   DOW_RET  GOLD_RET   OIL_RET  USDINR_RET  \
Date                                                                          
2023-01-03        NaN         NaN       NaN       NaN       NaN         NaN   
2023-01-04   0.007539    0.006911  0.004026  0.007121 -0.053165    0.000964   
2023-01-05  -0.011646   -0.014679 -0.010210 -0.009715  0.011395   -0.001440   
2023-01-06   0.022841    0.025623  0.021273  0.016023  0.001357   -0.000258   
2023-01-09  -0.000768    0.006279 -0.003359  0.004560  0.011658   -0.004499   

             VIX_RET  NIFTY_RET  
Date                             
2023-01-03       NaN        NaN  
2023-01-04 -0.038865  -0.010399  
2023-01-05  0.020445  -0.002815  
2023-01-06 -0.059216  -0.007376  
2023-01-09  0.039754   0.013536  


## 5. Clean and Prepare Final Dataset

In [6]:
def prepare_final_dataset(returns: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare final dataset by:
    - Resetting index to make Date a column
    - Removing NaN values from pct_change()
    - Sorting by Date
    - Ensuring no duplicate dates
    - Ensuring Date is datetime type
    
    Args:
        returns: DataFrame with daily returns
    
    Returns:
        Cleaned DataFrame
    """
    # Reset index to make Date a column
    df = returns.reset_index()
    df = df.rename(columns={'index': 'Date', 'Date': 'Date'})
    
    # Ensure Date is datetime type
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Remove NaN values (first row from pct_change and any other NaN)
    initial_rows = len(df)
    df = df.dropna()
    rows_removed = initial_rows - len(df)
    print(f"Removed {rows_removed} rows with NaN values")
    
    # Sort by Date
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Remove duplicate dates if any
    duplicates = df.duplicated(subset=['Date']).sum()
    if duplicates > 0:
        print(f"Found {duplicates} duplicate dates, removing...")
        df = df.drop_duplicates(subset=['Date'], keep='first')
    else:
        print("No duplicate dates found")
    
    return df

# Prepare final dataset
final_df = prepare_final_dataset(returns_df)

Removed 1 rows with NaN values
No duplicate dates found


In [7]:
# Verify column order matches requirements
expected_columns = ['Date', 'SP500_RET', 'NASDAQ_RET', 'DOW_RET', 'GOLD_RET', 
                    'OIL_RET', 'USDINR_RET', 'VIX_RET', 'NIFTY_RET']

# Reorder columns to match expected order
final_df = final_df[expected_columns]

print("Final Dataset Columns:")
print(final_df.columns.tolist())

Final Dataset Columns:
['Date', 'SP500_RET', 'NASDAQ_RET', 'DOW_RET', 'GOLD_RET', 'OIL_RET', 'USDINR_RET', 'VIX_RET', 'NIFTY_RET']


## 6. Save Dataset to CSV

In [8]:
# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save to CSV
final_df.to_csv(OUTPUT_FILE, index=False)
print(f"Dataset saved to: {OUTPUT_FILE}")

Dataset saved to: ../Market_Data/raw\global_market_data.csv


## 7. Validation

In [9]:
# Reload and validate
validation_df = pd.read_csv(OUTPUT_FILE, parse_dates=['Date'])

print("=" * 60)
print("VALIDATION REPORT")
print("=" * 60)

# Dataset shape
print(f"\n1. Dataset Shape: {validation_df.shape}")
print(f"   - Rows: {validation_df.shape[0]}")
print(f"   - Columns: {validation_df.shape[1]}")

# Date range
print(f"\n2. Date Range:")
print(f"   - Start: {validation_df['Date'].min()}")
print(f"   - End: {validation_df['Date'].max()}")

# Missing values
print(f"\n3. Missing Values:")
missing = validation_df.isnull().sum()
print(missing.to_string())

# Data types
print(f"\n4. Data Types:")
print(validation_df.dtypes.to_string())

# First 5 rows
print(f"\n5. First 5 Rows:")
print(validation_df.head().to_string())

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

VALIDATION REPORT

1. Dataset Shape: (751, 9)
   - Rows: 751
   - Columns: 9

2. Date Range:
   - Start: 2023-01-04 00:00:00
   - End: 2025-12-31 00:00:00

3. Missing Values:
Date          0
SP500_RET     0
NASDAQ_RET    0
DOW_RET       0
GOLD_RET      0
OIL_RET       0
USDINR_RET    0
VIX_RET       0
NIFTY_RET     0

4. Data Types:
Date          datetime64[ns]
SP500_RET            float64
NASDAQ_RET           float64
DOW_RET              float64
GOLD_RET             float64
OIL_RET              float64
USDINR_RET           float64
VIX_RET              float64
NIFTY_RET            float64

5. First 5 Rows:
        Date  SP500_RET  NASDAQ_RET   DOW_RET  GOLD_RET   OIL_RET  USDINR_RET   VIX_RET  NIFTY_RET
0 2023-01-04   0.007539    0.006911  0.004026  0.007121 -0.053165    0.000964 -0.038865  -0.010399
1 2023-01-05  -0.011646   -0.014679 -0.010210 -0.009715  0.011395   -0.001440  0.020445  -0.002815
2 2023-01-06   0.022841    0.025623  0.021273  0.016023  0.001357   -0.000258 -0.059216  

In [10]:
# Additional statistics
print("\nReturn Statistics:")
print(validation_df.describe().round(6).to_string())


Return Statistics:
                                Date   SP500_RET  NASDAQ_RET     DOW_RET    GOLD_RET     OIL_RET  USDINR_RET     VIX_RET   NIFTY_RET
count                            751  751.000000  751.000000  751.000000  751.000000  751.000000  751.000000  751.000000  751.000000
mean   2024-07-02 23:36:59.440745728    0.000820    0.001153    0.000531    0.001194   -0.000193    0.000113    0.002426    0.000507
min              2023-01-04 00:00:00   -0.059750   -0.059681   -0.055026   -0.057352   -0.085680   -0.022850   -0.357539   -0.059294
25%              2023-10-03 12:00:00   -0.003456   -0.004991   -0.003809   -0.004160   -0.012685   -0.000927   -0.038877   -0.003396
50%              2024-07-03 00:00:00    0.000996    0.001646    0.000776    0.001406    0.000484    0.000044   -0.005552    0.000181
75%              2025-04-02 12:00:00    0.005812    0.008149    0.005189    0.007157    0.012436    0.001087    0.032247    0.004756
max              2025-12-31 00:00:00    0.095154 

## Summary

This notebook:
1. Downloaded Close prices for 8 global market proxies from Yahoo Finance
2. Calculated daily returns using `pct_change()`
3. Cleaned the data (removed NaN, sorted by date, verified no duplicates)
4. Saved the final dataset to `Market_Data/raw/global_market_data.csv`

**Important Note for Merging:**
- Global market data should be lagged by 1 day when merging with stock data to avoid data leakage
- This lagging will be done in the data merging notebook (08_data_merging_and_alignment.ipynb)